In [112]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [113]:
%autoreload 2
from utils import (
    select_users_by_period,
    create_hourly_user_dataset,
)
from visualization_utils import (
    plot_user_metrics,
    plot_market_features,
)


In [213]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_columns', 500)

# MARKET = "eth_cbbtc_usdc"
MARKET = "base_cbbtc_usdc_full"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

df.head(2)

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,event_sequence_type,collateral_asset_symbol,loan_asset_symbol
0,0x4b95f45c3afdbdb97aca807ddc82d4d2b7cd2c09801b...,MarketSupply,1726217265,0xf8b9D83574574A345c32905a2A62D1d84650A0Ae,1000000,1.001,0,0.0,base_cbbtc_usdc,2024-09-13 08:47:45,0x9103c3b4e834476c9a62ea009ba2c884ee42e94e6e31...,0.0,0.0,1.000000,0.0,0.0,0.000000,1,0.002968,0.0,0.002968,0.000000,58132.0,1,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,1.0,0.000000,0.0,0.000000,loan_position_supply,False,0.001659,-0.002464,-0.00422,0.003227,-0.002464,-0.000207,loan_position_supply,cbBTC,USDC
1,0xbc8e2030283aef18ec38474c5a6f62dbb20bc9d18688...,MarketBorrow,1726217705,0xf8b9D83574574A345c32905a2A62D1d84650A0Ae,1000000,1.001,0,0.0,base_cbbtc_usdc,2024-09-13 08:55:05,0x9103c3b4e834476c9a62ea009ba2c884ee42e94e6e31...,1.0,0.0,1.016478,1.0,0.0,0.983789,3,0.002968,0.0,0.041709,0.041041,58132.0,1,0.0,0.0,0.0,1.0,0.0,0.0001,5.8132,1.0,1.0,0.172022,0.0,4.999352,position_open,False,0.001659,-0.002464,-0.00422,0.003227,-0.002464,-0.000207,position_open,cbBTC,USDC


In [217]:
df["utilization_after"].quantile(0.98)


0.9042481725864738

In [196]:
# market_df["borrow_rate_rolling"].plot()

plot_market_features(
    market_df,
    ["borrow_rate", "utilization"]
)

In [145]:
import pandas as pd

import numpy as np
import pandas as pd


import numpy as np
import pandas as pd

def detect_market_spikes(df, start_date, lookback_hours=6, metrics_config=None, actions_limit=10, max_recovery_events=50):
    """
    Detect spikes in specified market metrics and return enriched spike information.
    
    Returns list of dicts with keys:
        trigger_datetime, recovery_datetime (if recovered), recovery_time_seconds,
        spike_magnitudes (dict), trigger_event_types (comma-separated),
        market_state (dict with total_borrow, total_supply, collateral_price, loan_asset_price,
                     debt_before, supply_before, utilization_before),
        actions_df (DataFrame of actions during spike), spike_duration_events (int).
    """
    if metrics_config is None:
        metrics_config = {
            'utilization_after': {'spike_threshold': 0.05, 'high_threshold': 0.9, 'tolerance': 0.02}
        }
    
    if 'datetime' not in df.columns and 'timestamp' in df.columns:
        df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
    df = df.sort_values('timestamp').copy()
    
    start_ts = pd.Timestamp(start_date).timestamp()
    df = df[df['timestamp'] >= start_ts].reset_index(drop=True)
    if df.empty:
        return []
    
    timestamps = df['timestamp'].values
    lookback_sec = lookback_hours * 3600
    past_indices = np.searchsorted(timestamps, timestamps - lookback_sec, side='right') - 1
    
    metric_arrays = {}
    for metric in metrics_config.keys():
        if metric in df.columns:
            metric_arrays[metric] = df[metric].values
        else:
            raise KeyError(f"Metric '{metric}' not found in DataFrame columns")
    
    spikes = []
    i = 0
    n = len(df)
    while i < n and len(spikes) < actions_limit:
        past_idx = past_indices[i]
        if past_idx >= 0:
            triggered_metrics = {}
            baseline_vals = {}
            for metric, cfg in metrics_config.items():
                current_val = metric_arrays[metric][i]
                past_val = metric_arrays[metric][past_idx]
                delta = current_val - past_val
                if delta > cfg['spike_threshold'] and current_val > cfg['high_threshold']:
                    triggered_metrics[metric] = delta
                    baseline_vals[metric] = past_val
            
            if triggered_metrics:
                trigger_idx = i
                trigger_row = df.iloc[trigger_idx]
                trigger_tx_hash = trigger_row['hash']
                # Get all rows with same hash (same transaction) at trigger time
                trigger_tx_rows = df[(df['timestamp'] == trigger_row['timestamp']) & (df['hash'] == trigger_tx_hash)]
                trigger_event_types = ','.join(sorted(trigger_tx_rows['type'].unique()))
                
                # Market state at trigger (use before values or after? using before for consistency)
                market_state = {
                    'total_borrow': trigger_row.get('total_borrow_before', np.nan),
                    'total_supply': trigger_row.get('total_supply_before', np.nan),
                    'collateral_price': trigger_row.get('collateral_price', np.nan),
                    'loan_asset_price': trigger_row.get('loan_asset_price', np.nan),
                    'debt_before': trigger_row.get('debt_before', np.nan),
                    'supply_before': trigger_row.get('supply_before', np.nan),
                    'utilization_before': trigger_row.get('utilization_before', np.nan)
                }
                
                # Find recovery
                recovery_idx = None
                recovery_time = None
                for offset in range(max_recovery_events):
                    j = trigger_idx + offset
                    if j >= n:
                        break
                    all_recovered = True
                    for metric, cfg in metrics_config.items():
                        if metric in triggered_metrics:
                            current = metric_arrays[metric][j]
                            baseline = baseline_vals[metric]
                            # Recovery if: back to baseline (± tolerance) OR below high_threshold - tolerance
                            high_thresh = cfg['high_threshold']
                            if not (abs(current - baseline) <= cfg['tolerance'] or current < high_thresh - cfg['tolerance'] / 2):
                                all_recovered = False
                                break
                    if all_recovered:
                        recovery_idx = j
                        recovery_time = df.iloc[recovery_idx]['timestamp'] - df.iloc[trigger_idx]['timestamp']
                        break
                
                if recovery_idx is not None:
                    spike_df = df.iloc[trigger_idx:recovery_idx+1].copy()
                    next_i = recovery_idx + 1
                else:
                    # No recovery within limit, take up to max_recovery_events rows
                    end_idx = min(trigger_idx + max_recovery_events, n)
                    spike_df = df.iloc[trigger_idx:end_idx].copy()
                    recovery_time = None
                    next_i = end_idx
                
                spikes.append({
                    'trigger_datetime': trigger_row['datetime'],
                    'recovery_datetime': df.iloc[recovery_idx]['datetime'] if recovery_idx is not None else None,
                    'recovery_time_seconds': recovery_time,
                    'spike_magnitudes': triggered_metrics,
                    'trigger_event_types': trigger_event_types,
                    'market_state': market_state,
                    'actions_df': spike_df,
                    'spike_duration_events': len(spike_df)
                })
                i = next_i
                continue
        i += 1
    
    return spikes


In [146]:
metrics_config = {
    'utilization_after': {
        'spike_threshold': 0.05,    # 5% increase
        'high_threshold': 0.89,      # must exceed 90% utilization
        'tolerance': 0.01           # recovery within 2% of baseline
    },
    'borrow_rate_after': {
        'spike_threshold': 0.03,    # 1% increase
        'high_threshold': 0.08,      # must exceed 10% borrow rate
        'tolerance': 0.01          # recovery within 0.5% of baseline
    }
}

# Detect spikes
spikes = detect_market_spikes(
    df=df,
    start_date='2025-03-01',
    lookback_hours=1,
    metrics_config=metrics_config,
    actions_limit=1000
)
len(spikes)

191

In [147]:
spikes[0]

{'trigger_datetime': '2025-03-07 12:47:11',
 'recovery_datetime': '2025-03-07 13:00:23',
 'recovery_time_seconds': 792,
 'spike_magnitudes': {'utilization_after': 0.10913165579713346},
 'trigger_event_types': 'MarketWithdraw',
 'market_state': {'total_borrow': 27921953.158088267,
  'total_supply': 35165182.39730424,
  'collateral_price': 88955.0,
  'loan_asset_price': 0.999983,
  'debt_before': 0.0,
  'supply_before': 27775405.24527457,
  'utilization_before': 0.7940227024168305},
 'actions_df':                                                   hash            type  \
 165  0x771c4d1ebba435393ad18e82e739dcdc681ce4e810fe...  MarketWithdraw   
 166  0xa4e7bd098aef1f0baa3014a5aa631fd004e1d48f7b7f...    MarketSupply   
 167  0x7156d8d30395312358e2f2abdae595de06ee76fc2be4...    MarketSupply   
 
       timestamp                                user_address         assets  \
 165  1741351631  0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB  4249113008361   
 166  1741351859  0xBEEF01735c132Ada46AA

In [130]:
for i in spikes:
    eps=0.001
    actions_df = i["actions_df"]
    actions_df = actions_df[np.abs(actions_df["borrow_rate_after"] - actions_df["borrow_rate_before"]) > eps]
    print(i["trigger_datetime"], i["actions_df"]["datetime"].max(), i["actions_df"].shape[0], "->", actions_df.shape[0])
    print(actions_df["type"].value_counts())

2025-03-07 12:47:11 2025-03-07 13:00:23 4 -> 3
type
MarketSupply      2
MarketWithdraw    1
Name: count, dtype: int64
2025-03-11 20:17:59 2025-03-12 12:46:59 50 -> 32
type
MarketSupply              16
MarketWithdraw            12
MarketBorrow               3
MarketSupplyCollateral     1
Name: count, dtype: int64
2025-03-12 12:49:59 2025-03-12 13:00:35 3 -> 3
type
MarketSupply      2
MarketWithdraw    1
Name: count, dtype: int64
2025-03-13 23:46:59 2025-03-14 00:00:35 4 -> 3
type
MarketSupply      2
MarketWithdraw    1
Name: count, dtype: int64
2025-03-14 01:46:59 2025-03-14 02:00:35 6 -> 5
type
MarketWithdraw    3
MarketSupply      2
Name: count, dtype: int64
2025-03-14 03:46:59 2025-03-14 04:04:59 6 -> 5
type
MarketWithdraw    3
MarketSupply      2
Name: count, dtype: int64
2025-03-14 05:47:11 2025-03-14 09:50:47 13 -> 9
type
MarketSupply      5
MarketWithdraw    4
Name: count, dtype: int64
2025-03-14 19:46:59 2025-03-14 20:13:23 9 -> 5
type
MarketSupply      3
MarketWithdraw    2
Nam

In [247]:
spikes[0].keys()
spikes[0]["trigger_event_types"]

'MarketWithdraw'

In [244]:
spikes[0]["actions_df"][[
    "datetime",
    "type",
    "utilization_before",
    "utilization_after",
    "user_address"
]]

,datetime,type,utilization_before,utilization_after,user_address
165,2025-03-07 12:47:11,MarketWithdraw,0.794023,0.903154,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB
166,2025-03-07 12:50:59,MarketSupply,0.903154,0.899974,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB
167,2025-03-07 13:00:23,MarketSupply,0.899974,0.789947,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB


**Spikes actions analisys**

In [232]:
def analyze_action_impact(spikes, delta_threshold=-0.01):
    """
    For each action in each spike (excluding the trigger), compute delta_util and volume.
    Returns:
      - all_actions: DataFrame of every action
      - summary: aggregated by action type (overall)
      - spike_summary: per‑spike occurrence and impact of each action type
    """
    action_records = []
    spike_records = []
    
    for spike in spikes:
        df_actions = spike['actions_df'].copy().dropna()
        if df_actions.empty:
            continue
        
        trigger_ts = spike['trigger_datetime']
        df_actions = df_actions[df_actions['datetime'] > trigger_ts]
        if df_actions.empty:
            continue
        
        spike_id = spike['trigger_datetime']
        
        # Per‑action calculations
        df_actions['delta_util'] = df_actions['utilization_after'] - df_actions['utilization_before']
        if 'assets_usd' in df_actions.columns:
            df_actions['volume_usd'] = df_actions['assets_usd'].abs()
        elif 'assets' in df_actions.columns and 'collateral_price' in df_actions.columns:
            df_actions['volume_usd'] = df_actions['assets'].abs() * df_actions['collateral_price']
        else:
            df_actions['volume_usd'] = 1.0
        
        # Store for per‑action summary
        action_records.append(df_actions[['type', 'delta_util', 'volume_usd', 'timestamp']])
        
        # Per‑spike summary: which action types occurred and had significant effect
        action_types = df_actions['type'].unique()
        for atype in action_types:
            type_actions = df_actions[df_actions['type'] == atype]
            n_actions = len(type_actions)
            total_volume = type_actions['volume_usd'].sum()
            has_significant = (type_actions['delta_util'] < delta_threshold).any()
            # Also compute the best (most negative) delta achieved by this action type in this spike
            best_delta = type_actions['delta_util'].min()
            
            spike_records.append({
                'spike_id': spike_id,
                'action_type': atype,
                'occurred': True,
                'n_actions': n_actions,
                'total_volume_usd': total_volume,
                'has_significant_reduction': has_significant,
                'best_delta': best_delta
            })
    
    if not action_records:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    
    all_actions = pd.concat(action_records, ignore_index=True)
    all_actions = all_actions[all_actions["type"].isin([
        "MarketSupply",
        "MarketRepay",
        "MarketLiquidation",
        "MarketBorrow",
        "MarketWithdraw",
    ])]

    
    # Overall summary (same as before)
    summary = all_actions.groupby('type').agg(
        count=('delta_util', 'size'),
        mean_delta=('delta_util', 'mean'),
        median_delta=('delta_util', 'median'),
        total_volume_usd=('volume_usd', 'sum'),
        weighted_mean_delta=('delta_util', lambda x: np.average(x, weights=all_actions.loc[x.index, 'volume_usd']))
    ).round(4)
    
    # Per‑spike summary aggregated across spikes
    spike_df = pd.DataFrame(spike_records)
    if not spike_df.empty:
        spike_summary = spike_df.groupby('action_type').agg(
            spikes_present=('occurred', 'sum'),
            spikes_present_pct=('occurred', lambda x: x.sum() / len(spikes) * 100),
            total_actions=('n_actions', 'sum'),
            avg_actions_per_spike=('n_actions', 'mean'),
            avg_volume_per_spike=('total_volume_usd', 'mean'),
            spikes_with_significant_reduction=('has_significant_reduction', 'sum'),
            pct_spikes_with_significant=('has_significant_reduction', lambda x: x.sum() / len(spikes) * 100),
            avg_best_delta=('best_delta', 'mean')
        ).round(4)
    else:
        spike_summary = pd.DataFrame()
    
    return all_actions, summary, spike_summary


analyze_action_impact(spikes)[1]





,count,mean_delta,median_delta,total_volume_usd,weighted_mean_delta
type,,,,,
MarketBorrow,59,-0.0012,0.0000,2.611209e+07,0.0203
MarketLiquidation,4,-0.0008,-0.0005,7.575848e+05,-0.0017
MarketRepay,40,-0.0035,-0.0001,3.826432e+07,-0.0242
MarketSupply,1027,-0.0121,-0.0014,1.813664e+09,-0.0520
MarketWithdraw,485,0.0044,0.0005,2.924623e+08,0.0271


In [192]:
for i in spikes[:1550]:
    df_actions = i['actions_df'].copy()
    if df_actions.empty:
        continue
    
    trigger_ts = i['trigger_datetime']
    df_actions = df_actions[df_actions['datetime'] > trigger_ts]
    if df_actions.empty:
        continue
    if 'MarketRepay' in df_actions["type"].unique():
        display(df_actions[df_actions['type'] == 'MarketRepay'][[
            "datetime",
            "type",
            "utilization_before",
            "utilization_after",
            "borrow_rate_after",
            "user_address"
        ]])
        print("=" * 250)


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
537,2025-03-15 09:12:23,MarketRepay,0.910516,0.908965,0.057561,0x44B774D29e6E06B57975B04Ece9e3FcEf0A3137E


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
1914,2025-04-03 13:37:35,MarketRepay,0.903896,0.903865,0.045339,0x010B23B2f2A5F6bC4ce0c68A5f65c248c1129831


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
2605,2025-04-10 03:02:11,MarketRepay,0.930217,0.923235,0.069528,0xa77D05335D7C1a913C7708FB4f22340bE3ABEd55


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
3226,2025-04-13 20:36:59,MarketRepay,0.911575,0.911564,0.057285,0xA96c774DbCDad89eE7872078E58D5168170d684e


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
4343,2025-04-24 04:23:35,MarketRepay,0.909076,0.908673,0.051086,0xaD78662714CD0042B4c602a8e18d0e4a25cF79D7
4355,2025-04-24 06:29:59,MarketRepay,0.907895,0.907872,0.050165,0xaD78662714CD0042B4c602a8e18d0e4a25cF79D7


...

,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
32975,2025-11-06 21:25:11,MarketRepay,0.950639,0.949713,0.108903,0xBA10248Ac4DCC76ef27b58Aad637F02A27FBc4f6
32979,2025-11-06 21:28:11,MarketRepay,0.949709,0.949550,0.108689,0xBA10248Ac4DCC76ef27b58Aad637F02A27FBc4f6
32981,2025-11-06 21:29:23,MarketRepay,0.949550,0.948669,0.107535,0x559458Aac63528fB18893d797FF223dF4D5fa3C9
32993,2025-11-06 21:44:11,MarketRepay,0.952147,0.950473,0.109900,0x559458Aac63528fB18893d797FF223dF4D5fa3C9
33000,2025-11-06 21:56:47,MarketRepay,0.951973,0.950137,0.109460,0x559458Aac63528fB18893d797FF223dF4D5fa3C9
33011,2025-11-06 22:04:35,MarketRepay,0.959677,0.956170,0.117796,0x559458Aac63528fB18893d797FF223dF4D5fa3C9


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
33039,2025-11-06 22:41:23,MarketRepay,0.939878,0.921386,0.072016,0xbCd16D361adf5E7e0dc0010756B49666b4f38eE0
33053,2025-11-06 23:16:47,MarketRepay,0.913929,0.913911,0.062363,0x1A92B816dF51c0b73bf96cc2f1fd8eBC1ed164Dd


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
50613,2025-12-31 15:15:35,MarketRepay,0.929143,0.929143,0.069997,0x6344fEc307f50466646338bc1c4EaF41A38cb93D
50619,2025-12-31 15:23:23,MarketRepay,0.929006,0.928724,0.069528,0x9f127B66B1620D97de98746C27e245612E40285c


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
54713,2026-01-09 16:37:11,MarketRepay,0.914181,0.91418,0.058355,0xd097d150EDB4a8208915B46e3777b14a0d6bc136


,datetime,type,utilization_before,utilization_after,borrow_rate_after,user_address
56280,2026-01-14 16:31:23,MarketRepay,0.886904,0.886822,0.038937,0xc92f38f5946347c170b0f91D5dDd0f02405c3329


**Modeling -- spike duration**

In [168]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import statsmodels.api as sm
from sklearn.feature_selection import f_regression



def extract_spike_features(spikes):
    features_list = []
    for spike in spikes:
        df_actions = spike['actions_df']
        valid_actions = ['MarketRepay', 'MarketSupply', 'MarketWithdraw']
        df_actions = df_actions[df_actions['type'].isin(valid_actions)]
        if df_actions.empty:
            continue
        
        spike_mag = spike['spike_magnitudes'].get('utilization_after', 0)
        market_state = spike['market_state']
        
        # Use raw counts instead of proportions
        action_counts = df_actions['type'].value_counts().to_dict()
        # Drop one category to avoid perfect multicollinearity (e.g., drop MarketWithdraw)
        drop_col = 'MarketWithdraw'
        norm_counts = {}
        for k, v in action_counts.items():
            if k != drop_col:
                norm_counts[f'count_{k}'] = v
        
        # First action after trigger (use raw count, not one-hot)
        if len(df_actions) > 1:
            first_action = df_actions.iloc[1]['type']
        else:
            first_action = 'none'
        
        n_unique_users = df_actions['user_address'].nunique()
        total_actions = len(df_actions)
        recovery_time = spike.get('recovery_time_seconds', np.nan)
        if pd.isna(recovery_time):
            continue
        
        features = {
            'spike_magnitude': spike_mag,
            'collateral_price': market_state.get('collateral_price', 0),
            'loan_asset_price': market_state.get('loan_asset_price', 0),
            'utilization_before': market_state.get('utilization_before', 0),
            'first_action_MarketSupply': 1 if first_action == 'MarketSupply' else 0,
            'first_action_MarketRepay': 1 if first_action == 'MarketRepay' else 0,
            **norm_counts,
            'recovery_time_seconds': recovery_time
        }
        features_list.append(features)
    
    df_features = pd.DataFrame(features_list).fillna(0)
    # Drop any remaining columns that are constant or nearly constant
    constant_cols = [col for col in df_features.columns if df_features[col].nunique() <= 1]
    df_features = df_features.drop(columns=constant_cols)
    return df_features

def train_recovery_model(features_df, target_col='recovery_time_seconds', test_size=0.2, use_ridge=True):
    """
    Train a linear regression model to predict recovery time with detailed feature analysis.
    Returns model, scaler, evaluation metrics, and feature importance DataFrame.
    """
    X = features_df.drop(columns=[target_col])
    y = np.log1p(features_df[target_col])
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train model with regularization to handle multicollinearity
    if use_ridge:
        model = Ridge(alpha=1.0)
        model.fit(X_train_scaled, y_train)
    else:
        model = LinearRegression()
        model.fit(X_train_scaled, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test_scaled)
    y_pred_exp = np.expm1(y_pred)
    y_test_exp = np.expm1(y_test)
    
    print("=" * 60)
    print("MODEL PERFORMANCE")
    print("=" * 60)
    print(f"R² score: {r2_score(y_test, y_pred):.3f}")
    print(f"RMSE (seconds): {np.sqrt(mean_squared_error(y_test_exp, y_pred_exp)):.1f}")
    print(f"MAE (seconds): {mean_absolute_error(y_test_exp, y_pred_exp):.1f}")
    
    # Feature importance using statsmodels for p-values
    X_train_sm = sm.add_constant(X_train_scaled)
    try:
        sm_model = sm.OLS(y_train, X_train_sm).fit()
        p_values = sm_model.pvalues[1:]  # exclude constant
    except:
        p_values = [np.nan] * len(X.columns)
    
    # Create feature importance DataFrame
    coef_df = pd.DataFrame({
        'feature': X.columns,
        'coefficient': model.coef_,
        'abs_coefficient': np.abs(model.coef_),
        'p_value': p_values,
        'significant': p_values < 0.05 if not np.isnan(p_values).any() else False
    })
    
    # For Ridge, we don't have p-values, so we use coefficient magnitude as importance
    if use_ridge:
        coef_df['importance'] = np.abs(model.coef_)
        coef_df = coef_df.sort_values('importance', ascending=False)
        print("\n" + "=" * 60)
        print("TOP 10 FEATURES (Ridge Regression - by coefficient magnitude)")
        print("=" * 60)
    else:
        coef_df = coef_df.sort_values('abs_coefficient', ascending=False)
        print("\n" + "=" * 60)
        print("TOP 10 FEATURES (Linear Regression - by coefficient magnitude)")
        print("=" * 60)
        print("Positive coefficient = slower recovery | Negative = faster recovery")
        print("p < 0.05 indicates statistical significance")
    
    print(coef_df.head(10).to_string(index=False))
    
    # Also show F-regression scores for feature ranking
    f_scores, f_pvalues = f_regression(X_train_scaled, y_train)
    f_df = pd.DataFrame({
        'feature': X.columns,
        'f_score': f_scores,
        'f_pvalue': f_pvalues
    }).sort_values('f_score', ascending=False)
    
    print("\n" + "=" * 60)
    print("TOP 10 FEATURES (F-regression - univariate predictive power)")
    print("=" * 60)
    print(f_df.head(10).to_string(index=False))
    
    return model, scaler, coef_df


spikes_features_df = extract_spike_features(spikes)
spikes_features_df
train_recovery_model(
    spikes_features_df
)

MODEL PERFORMANCE
R² score: 0.405
RMSE (seconds): 12200.6
MAE (seconds): 3229.5

TOP 10 FEATURES (Ridge Regression - by coefficient magnitude)
                  feature  coefficient  abs_coefficient      p_value  significant  importance
       count_MarketSupply     1.026258         1.026258 9.082526e-17         True    1.026258
         collateral_price    -0.374613         0.374613 3.933650e-03         True    0.374613
        count_MarketRepay     0.355592         0.355592 8.490802e-04         True    0.355592
first_action_MarketSupply    -0.234635         0.234635 2.403798e-02         True    0.234635
         loan_asset_price     0.148737         0.148737 2.394057e-01        False    0.148737
       utilization_before     0.082317         0.082317 5.331117e-01        False    0.082317
          spike_magnitude    -0.049719         0.049719 6.949281e-01        False    0.049719

TOP 10 FEATURES (F-regression - univariate predictive power)
                  feature   f_score     f_p

(Ridge(),
 StandardScaler(),
                       feature  coefficient  abs_coefficient       p_value  \
 x6         count_MarketSupply     1.026258         1.026258  9.082526e-17   
 x2           collateral_price    -0.374613         0.374613  3.933650e-03   
 x7          count_MarketRepay     0.355592         0.355592  8.490802e-04   
 x5  first_action_MarketSupply    -0.234635         0.234635  2.403798e-02   
 x3           loan_asset_price     0.148737         0.148737  2.394057e-01   
 x4         utilization_before     0.082317         0.082317  5.331117e-01   
 x1            spike_magnitude    -0.049719         0.049719  6.949281e-01   
 
     significant  importance  
 x6         True    1.026258  
 x2         True    0.374613  
 x7         True    0.355592  
 x5         True    0.234635  
 x3        False    0.148737  
 x4        False    0.082317  
 x1        False    0.049719  )

**Analyzing users with huge volume**

In [101]:
user_debt = df[df["user_address"].isin( df[df["type"] == 'MarketBorrow']["user_address"].unique() )].groupby("user_address")["debt_after"].max()
user_debt.describe()

small_users = user_debt[user_debt<np.quantile(user_debt, 0.9)].index
large_users = user_debt[user_debt>=np.quantile(user_debt, 0.9)].index
user_debt[large_users].describe()

count    5.600000e+01
mean     1.414328e+07
std      3.259749e+07
min      1.249702e+06
25%      2.436677e+06
50%      3.851237e+06
75%      9.353998e+06
max      2.014634e+08
Name: debt_after, dtype: float64

In [103]:
small_df = df[(df["datetime"] >= "2025-03-01") & df["user_address"].isin(small_users)]
(small_df["utilization_after"] - small_df["utilization_before"]).sort_values(ascending=False)

3210    0.016937
3211    0.016937
3213    0.015439
3212    0.015439
3215    0.013620
          ...   
7440   -0.118698
7240   -0.125235
5498   -0.137269
4019   -0.140800
3840   -0.172284
Length: 5901, dtype: float64

In [111]:
# small_df.iloc[3840]
small_df[small_df["user_address"] == "0xf37414de43b7D79859d30ec39C6bC0B3c7672135"][[
    "datetime",
    "type",
    "debt_before",
    "debt_after",
    "ltv_after",
    "event_type",
]]

,datetime,type,debt_before,debt_after,ltv_after,event_type
40186,2025-11-18 21:42:11,MarketSupplyCollateral,0.000000,0.000000,0.000000,collateral_add
40188,2025-11-18 21:44:59,MarketBorrow,0.000000,249928.137335,0.684797,borrow_more
40189,2025-11-18 21:53:59,MarketSupplyCollateral,249928.137335,249928.137335,0.406510,collateral_add
40191,2025-11-18 21:56:35,MarketBorrow,249928.137335,449870.647204,0.731718,borrow_more
40192,2025-11-18 21:59:35,MarketSupplyCollateral,449870.647204,449870.647204,0.552004,collateral_add
40193,2025-11-18 22:02:23,MarketBorrow,449870.647204,559839.027631,0.686938,borrow_more
40197,2025-11-18 22:04:47,MarketSupplyCollateral,559839.027631,559839.027631,0.605288,collateral_add
41007,2025-11-21 07:54:23,MarketWithdrawCollateral,559854.877203,559854.877203,0.782707,collateral_withdraw
41009,2025-11-21 07:59:11,MarketRepay,559854.877203,434419.554000,0.607342,repay_partial
41010,2025-11-21 08:01:11,MarketWithdrawCollateral,434419.554000,434419.554000,0.737870,collateral_withdraw


**All markets analisys**

In [249]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime

def process_all_markets(events_dir, hourly_dir, metrics_config=None, 
                        spike_params=None, action_threshold=-0.01, 
                        output_md_path="/Users/yegortrussov/Documents/ml/lending_protocols/experiments_logs/spikes_events/utilization_stabilize_report.md"):
    """
    Process all eligible markets, detect spikes, analyze actions, and produce a markdown report.
    """
    if metrics_config is None:
        metrics_config = {
            'utilization_after': {
                'spike_threshold': 0.05,
                'high_threshold': 0.9,
                'tolerance': 0.02
            }
        }
    if spike_params is None:
        spike_params = {
            'lookback_hours': 6,
            'actions_limit': 300,
            'max_recovery_events': 100
        }
    
    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_md_path), exist_ok=True)
    
    all_markets_data = []
    
    for fname in os.listdir(events_dir):
        if not fname.endswith('.csv'):
            continue
        market_name = fname[:-4]
        if 'PT-' in market_name:
            print(f"Skipping PT- market: {market_name}")
            continue
        
        events_path = os.path.join(events_dir, fname)
        
        try:
            df = pd.read_csv(events_path)
            df = df.sort_values('timestamp')
            cut_idx = int(len(df) * 0.2) 
            df = df.iloc[cut_idx:].reset_index(drop=True)

        except Exception as e:
            print(f"Error loading {market_name}: {e}")
            continue
        
        # Check last total_borrow
        if df.empty or df['total_borrow_after'].iloc[-1] < 10000:
            print(f"Skipping {market_name}: low total borrow ({df['total_borrow_after'].iloc[-1]:.0f})")
            continue
        
        # Compute market metrics
        max_total_supply = df['total_supply_after'].max()
        total_events = len(df)
        n_users = df['user_address'].nunique()
        
        # Ensure datetime column
        if 'datetime' not in df.columns and 'timestamp' in df.columns:
            df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
        
        # Detect spikes
        metrics_config["utilization_after"]["high_threshold"] = df["utilization_after"].quantile(0.98)
        spikes = detect_market_spikes(
            df,
            start_date=df['datetime'].min(),
            **spike_params,
            metrics_config=metrics_config
        )
        
        n_spikes = len(spikes)
        action_summary = None
        spike_occurrence = None
        
        if spikes:
            _, overall_summary, spike_summary = analyze_action_impact(spikes, delta_threshold=action_threshold)
            # Keep only the most relevant columns
            try:
                action_summary = overall_summary[['count', 'weighted_mean_delta']].sort_values('weighted_mean_delta').fillna(0)
                spike_occurrence = spike_summary[['spikes_present_pct', 'pct_spikes_with_significant', 'avg_best_delta']].fillna(0)
            except Exception as e:
                continue
        
        all_markets_data.append({
            'market': market_name,
            'max_total_supply': max_total_supply,
            'total_events': total_events,
            'n_users': n_users,
            'n_spikes': n_spikes,
            'action_summary': action_summary,
            'spike_occurrence': spike_occurrence,
            'spikes': spikes,
        })
    all_spikes = []
    for m in all_markets_data:
        if m['action_summary'] is not None and m["market"] not in ('eth_wbtc_usdt', 'base_wbtc_usdt', 'eth_weth_usdt'):
            # You need to store the actual spikes in all_markets_data first
            all_spikes.extend(m['spikes'])  # assuming you stored spikes
    
    # Sort markets by number of users descending
    all_markets_data.sort(key=lambda x: x['n_users'], reverse=True)
    
    # Generate markdown report
    with open(output_md_path, 'w') as f:
        f.write("# Market Spike Analysis Report\n\n")

        if all_spikes:
            _, global_action_summary, global_spike_summary = analyze_action_impact(all_spikes, delta_threshold=action_threshold)
            global_action_summary = global_action_summary.fillna(0)
            
            
            # Add to markdown report before per-market sections
            f.write("## Combined Analysis Across All Markets\n\n")
            f.write("### Action Impact (weighted by volume) – All Markets\n\n")
            f.write("| Action Type | Count | Weighted Mean ΔUtil |\n")
            f.write("|-------------|-------|---------------------|\n")
            for idx, row in global_action_summary.iterrows():
                f.write(f"| {idx} | {row['count']} | {row['weighted_mean_delta']:.4f} |\n")
            
            f.write("\n")


            pivot_data = []
            for m in all_markets_data:
                row = {'market': m['market'], 'n_spikes': m['n_spikes']}
                if m['action_summary'] is not None:
                    for action_type, row_data in m['action_summary'].iterrows():
                        row[action_type] = row_data['weighted_mean_delta']
                pivot_data.append(row)
            pivot_df = pd.DataFrame(pivot_data).set_index('market').fillna('')

            # In markdown, generate the table with bold formatting
            f.write("### Weighted Mean ΔUtil by Market and Action Type\n\n")
            f.write("| Market | N Spikes | " + " | ".join([col for col in pivot_df.columns if col != 'n_spikes']) + " |\n")
            f.write("|" + "|".join(["---"] * (len(pivot_df.columns)+1)) + "|\n")

            for market, row in pivot_df.iterrows():
                row_cells = [str(int(row['n_spikes']))]
                for col in [col for col in pivot_df.columns if col != 'n_spikes']:
                    val = row[col]
                    if val != '' and abs(val) == max([abs(v) for v in row if v != '' and v != row['n_spikes']], default=0):
                        row_cells.append(f"**{val:.4f}**")
                    else:
                        row_cells.append(f"{val:.4f}" if val != '' else "")
                f.write(f"| {market} | " + " | ".join(row_cells) + " |\n")


        
        # Summary over all markets
        f.write("## Summary Across All Markets\n\n")
        f.write("| Metric | Value |\n")
        f.write("|--------|-------|\n")
        total_markets = len(all_markets_data)
        total_spikes = sum(m['n_spikes'] for m in all_markets_data)
        total_users = sum(m['n_users'] for m in all_markets_data)
        total_events = sum(m['total_events'] for m in all_markets_data)
        f.write(f"| Number of markets processed | {total_markets} |\n")
        f.write(f"| Total spikes detected | {total_spikes} |\n")
        f.write(f"| Total unique users across all markets | {total_users} |\n")
        f.write(f"| Total events across all markets | {total_events} |\n\n")
        
        # Per-market sections
        f.write("## Per-Market Analysis\n\n")
        for m in all_markets_data:
            f.write(f"## {m['market']}\n\n")
            f.write(f"- **Max total supply (USD)**: {m['max_total_supply']:,.2f}\n")
            f.write(f"- **Total events**: {m['total_events']:,}\n")
            f.write(f"- **Number of users**: {m['n_users']:,}\n")
            f.write(f"- **Number of spikes**: {m['n_spikes']}\n\n")
            
            if m['n_spikes'] > 0 and m['action_summary'] is not None:
                f.write("#### Action Impact (weighted by volume)\n\n")
                f.write("| Action Type | Count | Weighted Mean ΔUtil\n")
                f.write("|-------------|-------|---------------------|\n")
                for idx, row in m['action_summary'].iterrows():
                    f.write(f"| {idx} | {row['count']} | {row['weighted_mean_delta']:.4f} |\n")

                trigger_counts = pd.Series([x["trigger_event_types"] for x in m["spikes"]]).value_counts().head(5)
                
                f.write("#### Top 5 Trigger Event Types\n\n")
                f.write("| Event Type | Count |\n")
                f.write("|------------|-------|\n")
                for event_type, count in trigger_counts.items():
                    f.write(f"| {event_type} | {count} |\n")

                f.write("\n")
                
                # f.write("\n#### Per‑Spike Occurrence\n\n")
                # f.write("| Action Type | % of spikes present | % spikes with significant reduction | Avg best ΔUtil |\n")
                # f.write("|-------------|---------------------|-------------------------------------|----------------|\n")
                # for idx, row in m['spike_occurrence'].iterrows():
                #     f.write(f"| {idx} | {row['spikes_present_pct']:.1f}% | {row['pct_spikes_with_significant']:.1f}% | {row['avg_best_delta']:.4f} |\n")
                f.write("\n")
            else:
                f.write("*No spikes detected or insufficient data for action analysis.*\n\n")
            
            f.write("---\n\n")
    
    print(f"Report saved to {output_md_path}")
    return all_markets_data

# Example usage
if __name__ == "__main__":
    EVENTS_DIR = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
    HOURLY_DIR = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
    OUTPUT_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/experiments_logs/spikes_events/utilization_stabilize_report.md"
    
    results = process_all_markets(EVENTS_DIR, HOURLY_DIR, output_md_path=OUTPUT_PATH)



Skipping PT- market: eth_PT-syrupUSDC-28AUG2025_usdc
Skipping PT- market: eth_PT-slvlUSD-25SEP2025_usdc
Skipping PT- market: eth_PT-USDe-27NOV2025_usds
Skipping PT- market: eth_PT-USDe-25SEP2025_dai
Skipping PT- market: eth_PT-syrupUSDC-30OCT2025_usdc
Skipping PT- market: eth_PT-stcUSD-23JUL2026_usdc
Skipping eth_slvlusd_usdc: low total borrow (1325)
Skipping PT- market: eth_PT-wstUSR-27MAR2025_usr
Skipping base_cbbtc_usdc_full1: low total borrow (0)
Skipping PT- market: eth_PT-sNUSD-5MAR2026_usdc
Skipping PT- market: PT-reUSD-25JUN2026_usdc
Skipping PT- market: eth_PT-USD0++-31OCT2024_usdc
Skipping PT- market: eth_PT-USDe-31JUL2025_dai
Skipping PT- market: eth_PT-USDe-25SEP2025_usdt
Skipping PT- market: eth_PT-USDe-25SEP2025_usdc
Skipping PT- market: eth_PT-wstUSR-27MAR2025_usdc
Skipping PT- market: eth_PT-slvlUSD-29MAY2025_usdc
Skipping PT- market: eth_PT-mHYPER-20NOV2025_usdc
Skipping eth_csusdl_usdc: low total borrow (6397)
Skipping PT- market: eth_PT-sdeUSD-1753142406_usdc
Skippin

In [225]:
# results["eth_wbtc_usdt"]
results
for i in results:
    if i["market"] == "eth_wbtc_usdt":
        display(i["spikes"])
        break

[{'trigger_datetime': '2025-02-03 04:29:35',
  'recovery_datetime': '2025-02-03 05:31:35',
  'recovery_time_seconds': 3720,
  'spike_magnitudes': {'utilization_after': 0.0696672035558632},
  'trigger_event_types': 'MarketWithdraw',
  'market_state': {'total_borrow': 40029697.54867299,
   'total_supply': 33559358.12627403,
   'collateral_price': 93866.0,
   'loan_asset_price': 1.001,
   'debt_before': 0.0,
   'supply_before': nan,
   'utilization_before': 1.1928028360391452},
  'actions_df':                                                  hash                    type  \
  10  0xdef6259057523d862c798ec447f14e1fcfaefb6e67be...          MarketWithdraw   
  11  0x43c4575d49a70acbc6aaa0c312b741efc4eda73444d6...  MarketSupplyCollateral   
  12  0x43c4575d49a70acbc6aaa0c312b741efc4eda73444d6...            MarketBorrow   
  13  0x8850516a70cb71c2cd91a1eaff51063f12b1b48fad07...          MarketWithdraw   
  14  0xec8e7cc9df7b602d9ae102437b3663754f44be66cb97...            MarketSupply   
  
     